# Stage 1: Multi-source ingest + obs schema standardisation

Load raw scRNA-seq data from one source dataset via `read_with_manifest`,
which applies obs schema standardisation (13 behaviors per SPEC), baseline QC metrics,
and writes a stage-1 checkpoint h5ad.

**What this notebook produces**:
- `obs` columns standardized across source datasets (Layer 1 core + Layer 2 CellxGene-aligned + Layer 3 project-defined)
- Baseline QC metrics (`n_genes`, `total_counts`, `pct_counts_mt`, `pct_counts_ribo`) computed on raw counts
- Gene symbols as `var.index` with Ensembl IDs in `var["ensembl_id"]`
- Stage 1 checkpoint `.h5ad` file ready for stage 2 QC

In [ ]:
# === PARAMS ===
# Edit these before running:
#   MANIFEST_PATH — which dataset to ingest
#   OUTPUT_PATH   — where to write the stage-1 checkpoint
#   RANDOM_SEED   — fixed for reproducibility

MANIFEST_PATH = "data/nowicki/manifest.yaml"   # path to per-dataset manifest
OUTPUT_PATH   = "results/nancang_stage1_loaded_v1.h5ad"
RANDOM_SEED   = 42

In [ ]:
# Ensure the framework src/ is on sys.path and CWD is set to the project root.
# Detects: if running from notebooks/ (Jupyter) or from project root (nbconvert).
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
print(f"PROJECT_ROOT: {_root}")


In [ ]:
# Imports (scanpy native API + framework functions only where a real gap exists).
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import yaml
import warnings

from scrna_integration import read_with_manifest

# Silence chatty warnings from mygene / scanpy during ingest.
warnings.filterwarnings("ignore", message=".*mygene.*")
warnings.filterwarnings("ignore", message=".*Layer2.*")

sc.settings.verbosity = 2  # show useful progress (0=quiet, 3=verbose)
import importlib.metadata; print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

In [ ]:
# Call read_with_manifest — the single framework entry point for multi-source ingest.
# This runs 13 standardisation steps (SPEC "The Three Functions") and returns a plain AnnData.
print(f"\n===== Ingesting from: {MANIFEST_PATH} =====\n")
adata = read_with_manifest(MANIFEST_PATH)

print(f"\nReturned AnnData: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
# Quick visual check: obs head, var head, and uns keys.
# PI / students inspect these to verify the manifest-driven schema is correct
# before proceeding to stage 2 QC.
print("=== obs.head() ===")
display(adata.obs.head())

print("\n=== var.head() ===")
display(adata.var.head())

print("\n=== obs columns ===")
print(list(adata.obs.columns))

print("\n=== uns keys ===")
print(list(adata.uns.keys()))

print(f"\n=== obs.dtypes sample ===")
print(adata.obs.dtypes.head(10))

In [ ]:
# Baseline QC metrics — already computed by read_with_manifest (step 12).
# These are per-cell metrics on raw counts, aligned across all source datasets
# regardless of author preprocessing state.
print("===== Baseline QC summary (raw counts) =====")
for col in ["n_genes", "total_counts", "pct_counts_mt", "pct_counts_ribo"]:
    if col in adata.obs.columns:
        vals = adata.obs[col]
        print(f"  {col:20s}: mean={vals.mean():.1f}  median={vals.median():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")
print(f"\n  source_dataset: {sorted(adata.obs['source_dataset'].unique())}")
n_samples = adata.obs['sample_id'].nunique() if "sample_id" in adata.obs.columns else "N/A"
print(f"  n_samples:         {n_samples}")

# Also check raw_matrix_path for SoupX (stage 2 needs this).
rp = adata.uns.get("raw_matrix_path", None)
print(f"\n  raw_matrix_path (for SoupX): {rp}")

In [ ]:
# Memory discipline self-check (one assertion before write — SPEC Memory Discipline).
# Guards the highest-impact memory regression: adata.X becoming dense or losing float32.
# If this ever fails, investigate which upstream operation densified or cast the matrix.
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# Write the stage checkpoint to disk.
# compression="lzf" is Memory Discipline #4 — faster than gzip,
# ~30% smaller than uncompressed, and preserves sparse CSR layout.
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

# Verify the file was written and is readable.
import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# Free memory across stage boundaries (Memory Discipline #3).
# Without this, the Jupyter kernel keeps the previous stage's AnnData
# alive when the next stage is run in the same kernel session.
del adata
import gc
gc.collect()
print("Memory released.")